# DADA2: Agentic RAG over SEC filings


In this assignmen, you will build an agentic RAG system over SEC filings. The corpus is intentionally complicated as it includes nine 10-K filings from three chip companies (NVIDIA, AMD, Micron) that all talk about the same things.

The assignment requires you to use a **chat model** to write queries, plans, and answers and a Jev-like **decision model** to classify questions and determine whether retrieved passages answer them. Unlike the chat models (your typical LLMs), the decision model will return a probability for each option instead of text.

The notebooks will guide you through the implementation, which you will later turn into a deployable app. Go through each notebook sequentially. Only after completing the notebooks, should you put together your app.

## The notebooks

| | | points |
|---|---|---|
| `DADA2_01_Corpus_and_Index` | 27 filings into a filterable Pinecone index | 4 |
| `DADA2_02_Retrieval_Strategies` | four strategies, each one measured | 6 |
| `DADA2_03_Agent` | a router, a diagnosis, and the graph | 6 |
| `DADA2_04_Evaluation` | two models and two agents, compared | 4 |

## What you are given

```
corpus/
  sections/      27 .txt files, the Item 1 / 1A / 7 sections of 9 filings
  financials/    3 .json files, XBRL figures from EDGAR
  filings/       1 raw .htm filing, for the optional part of notebook 01
  manifest.csv   one row per section file: company, year, section
eval/
  evalset.json        14 specific questions, one gold file each
  evalset_hard.json   10 comparative questions, several gold files each
rag/
  config.py      settings. Given.
  evaluate.py    the metrics and the comparison tables. Given.
  decide.py      the decision model. Given.
  ladder.py      the agent you will compare yours against. Given.
  util.py        retry and text_of. Given.
  retrieval.py   notebook 02 fills this in
  agent.py       notebook 03 fills this in
app.py           a Gradio front end over your agent. Given.
figures/architecture.png
```

Write in the notebooks first. When something works, move it into the matching file under `rag/`. Later notebooks and `app.py` import from `rag/`, so if a notebook runs while the module is still empty, you skipped that step.

## Why these filings

The corpus includes the filings for NVIDIA, AMD and Micron, three fiscal years each, split into the three sections that are mostly prose: Business, Risk Factors, and Management's Discussion.

All three sell into the same market, so all three write about export controls, HBM supply, TSMC and AI demand, in similar words, every year. A question about NVIDIA's 2026 risk factors has twenty-six other files full of near misses. Plain similarity search does badly here, and that is what makes the difference between strategies show up in the numbers.

## Setup

Three sets of keys. Ollama only if you want the local embedding backup.

```bash
cp example.env .env        # then fill it in
uv pip install rank-bm25 gradio
ollama pull qwen3-embedding:0.6b    # optional: the local embedding backup
```

**Gemini** is the chat model. Free tier: 15 requests a minute, 500 a day.

**Pinecone** holds the index.

**Cloudflare** hosts the decision model and the embedding model. You need an API token with the Workers AI permission; `example.env` says where to find it. The free tier is 10,000 neurons a day; indexing the whole corpus costs about 400 and a decision call a handful, so you will not run out.

Run the check below before you do anything else.

In [ ]:
import os, csv, json, importlib.util
from dotenv import load_dotenv

load_dotenv()

ok = True

for key in ("GOOGLE_API_KEY", "PINECONE_API_KEY", "CLOUDFLARE_API_TOKEN"):
    value = os.getenv(key, "")
    if not value or value.startswith("your_"):
        print(f"MISSING  {key}")
        ok = False
    else:
        print(f"ok       {key}")

for package in ("langchain_ollama", "langchain_pinecone", "pinecone", "langgraph",
                "rank_bm25", "requests"):
    if importlib.util.find_spec(package) is None:
        print(f"MISSING  {package}")
        ok = False
    else:
        print(f"ok       {package}")

rows = list(csv.DictReader(open("corpus/manifest.csv")))
print(f"ok       manifest lists {len(rows)} files, corpus/sections holds {len(os.listdir('corpus/sections'))}")
for which in ("evalset.json", "evalset_hard.json"):
    print(f"ok       eval/{which}: {len(json.load(open(os.path.join('eval', which))))} questions")

try:
    from rag import config
    dim = len(config.embeddings().embed_query("probe"))
    print(f"ok       embeddings via {config.EMBEDDINGS}, {dim} dimensions")
except Exception as e:
    print(f"MISSING  embeddings ({config.EMBEDDINGS}) did not answer: {type(e).__name__}")
    ok = False

try:
    from rag import decide
    choice, probs = decide.scope("What did Micron spend on R&D in fiscal 2025?")
    print(f"ok       decision model answered: {choice} ({probs[choice]:.2f})")
except Exception as e:
    print(f"MISSING  decision model: {e}")
    ok = False

print()
print("ready" if ok else "fix the MISSING lines first")

## Reranking: get a Cohere key now

Notebook 02 reranks with [Cohere's](https://docs.cohere.com/docs/rerank) `rerank-v3.5`, a cross-encoder with a free tier. Create a key and put it in `.env` as `COHERE_API_KEY`.


## Grading

The points are on the TODOs. Do the following while working on the assignment:

**Change one thing at a time.** Every row in your tables should differ from the row above it in one way.

**Say what you measured.** A number without its conditions is not a result. Specify which reranker, which k, which question set.

You are not graded on beating a target. Some strategies are supposed to lose on one of the question sets. Noticing that is worth more than claiming everything got better.

## Rate limits

Be mindful of the 500-a-day Gemini limit. Notebook 04 will involve you running both agents over a question set and the ladder agent spends about five chat calls per question. Run it once, when everything already works. Notebook 02 scores on a slice of six questions so you can iterate without spending the day's budget.

Pinecone's free tier allows a few indexes. Index the files once and reuse the index for vector search.

**Developing without spending the quota.** The chat model can run locally
while you build. Put `CHAT_BACKEND='ollama'` in `.env` and have Ollama running
with the Week 2 model (`ollama pull qwen3.5:4b`); every chat call then goes
to it instead of Gemini. It is slower and a little less accurate, so remove the
line for notebook 04, whose numbers should come from Gemini, and before you
deploy: Render does not set it and uses Gemini.
